### 公式化

`token_t = argmax P(token_t | token_1, token_2, ..., token_{t-1})`

### 直观理解

每一步生成的时候, **总是选取概率最大的候选 token ** 作为下一个 token, 然后继续生成, 直到达到终止条件或最大长度



In [30]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen3-0.6B'

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
)
model.eval()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [85]:
messages = [
    {'role': 'user', 'content': 'introduce yourself in 10 words'}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    enable_thinking = False,
    add_generation_prompt = True
)

text

'<|im_start|>user\nintroduce yourself in 10 words<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

In [86]:
inputs = tokenizer(text, return_tensors="pt",)
inputs

{'input_ids': tensor([[151644,    872,    198,    396,  47845,   6133,    304,    220,     16,
             15,   4244, 151645,    198, 151644,  77091,    198, 151667,    271,
         151668,    271]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

# 演示一次 model 的前向推理过程

In [87]:
# outputs = model(input_ids=inputs['input_ids'])
outputs = model(**inputs)

In [88]:
outputs_logits = outputs.logits
print(outputs_logits.shape)
outputs_logits


torch.Size([1, 20, 151936])


tensor([[[ 3.5938,  3.7812,  3.6562,  ...,  1.6484,  1.6484,  1.6484],
         [ 7.2812,  8.2500,  6.3125,  ...,  0.4141,  0.4141,  0.4141],
         [ 4.7188,  9.6250, 11.5625,  ...,  3.8281,  3.8281,  3.8281],
         ...,
         [ 5.1250, 17.0000,  6.9375,  ..., -0.3418, -0.3418, -0.3418],
         [-4.0312, -2.5469, -7.7812,  ..., -1.3516, -1.3516, -1.3516],
         [ 7.4062, 12.5625,  5.7188,  ...,  4.2812,  4.2812,  4.2812]]],
       dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>)

In [89]:
last_location_logits = outputs_logits[:, -1, :]
print(last_location_logits.shape)
last_location_logits

torch.Size([1, 151936])


tensor([[ 7.4062, 12.5625,  5.7188,  ...,  4.2812,  4.2812,  4.2812]],
       dtype=torch.bfloat16, grad_fn=<SelectBackward0>)

### 疑问: llm 为什么给前面的也有 logits

### 疑问: 那么我前面在看 agent 的时候, 其实是忽略了 token 的生成逻辑, 黑盒外又套了一个黑盒

### 步入正题

In [ ]:
# token选取方式定义
import torch
def greedy_decode(model, inputs, max_length, tokenizer=None):
    print(f"input: {repr(inputs)}")
    if tokenizer is not None:
        print(f'input_decode: {tokenizer.decode(inputs['input_ids'])}')
    for _ in range(max_length):
        outputs = model(**inputs)
        outputs_logits = outputs.logits[:, -1, :]
        next_token = torch.argmax(input=outputs_logits, dim=1, keepdim=True)
        next_attention = torch.ones(inputs['attention_mask'].shape[0], 1)
        inputs['input_ids'] = torch.cat([inputs['input_ids'], next_token], dim=1)
        inputs['attention_mask'] = torch.cat([inputs['attention_mask'], next_attention], dim=1)
        print(f'next_token: {next_token}')
        if tokenizer is not None:
            print(f'next_token_decode: {tokenizer.decode(next_token)}')
        if next_token.item() == model.config.eos_token_id:
            break
    return inputs['input_ids']


In [122]:
messages = [
    {'role': 'user', 'content': 'introduce yourself in 10 words'}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    enable_thinking = False,
    add_generation_prompt = True
)

inputs = tokenizer(
    text,
    return_tensors = 'pt'
)

output = greedy_decode(inputs=inputs, model=model, max_length=1000, tokenizer=tokenizer)

output

input: {'input_ids': tensor([[151644,    872,    198,    396,  47845,   6133,    304,    220,     16,
             15,   4244, 151645,    198, 151644,  77091,    198, 151667,    271,
         151668,    271]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


TypeError: argument 'ids': 'list' object cannot be interpreted as an integer

In [106]:
tokenizer.decode(output)

['<|im_start|>user\nintroduce yourself in 10 words<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nI am a language model designed to assist with questions and provide information.<|im_end|>']